# معمل LangGraph التفاعلي: بناء أول رسم بياني معتمد على الحالة
### Interactive Hands-on Lab: Stateful Multi-Actor Agent Graph (Lecture 68)

أهلاً بك في معمل التجارب التفاعلي لـ **LangGraph**!
في هذا المعمل، سننتقل من التفكير الخطي التقليدي إلى بناء **نظام معتمد على الحالة (Stateful Graph)** يتحرك بين العقد المختلفة بمرونة كاملة.

سنتعلم عملياً:
1. إعداد بيئة العمل والتحقق من المفاتيح.
2. تعريف بنية الحالة المشتركة (Graph State).
3. بناء عقد الجراف (Nodes) وتوصيلها بأسهم ثابتة وشرطية (Edges & Conditional Edges).
4. تشغيل وتتبع تدفق البيانات خطوة بخطوة.
5. استخدام الذاكرة ونقاط التفتيش (MemorySaver) واسترجاع تاريخ الجلسة.
6. قسم خاص: جرب بنفسك!


In [1]:
import os
import dotenv

dotenv.load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
print("جاهزية البيئة: تم العثور على مفتاح Groq بنجاح!" if api_key else "تنبيه: سنعمل باستخدام محاكي محلي لعدم وجود المفتاح.")


جاهزية البيئة: تم العثور على مفتاح Groq بنجاح!


In [2]:
from typing import Annotated, Dict, List, Literal, Any, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

print("تم استيراد كافة مكتبات LangGraph بنجاح!")


تم استيراد كافة مكتبات LangGraph بنجاح!


### 1. تعريف الحالة المشتركة (Graph State)
الحالة هي بمثابة الصندوق المشترك الذي تمرره العقد لبعضها.
لاحظ استخدام `add_messages`، هذا يضمن أن أي رسالة جديدة تضاف للقائمة بدلاً من مسح الرسائل القديمة.


In [3]:
class SimpleAgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    intent: str
    steps_count: int

print("تم تعريف كلاس الحالة SimpleAgentState بنجاح!")


تم تعريف كلاس الحالة SimpleAgentState بنجاح!


### 2. تجهيز نموذج المحادثة
سنستخدم ChatGroq مع نموذج سريع ومجاني لتوليد الإجابات الذكية.


In [4]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0) if api_key else None
print("النموذج جاهز للعمل:", type(llm).__name__ if llm else "MockModel")


النموذج جاهز للعمل:

 ChatGroq


### 3. بناء العقد (Nodes)
كل عقدة هي دالة بايثون بسيطة، تأخذ الحالة (state)، وتقوم بعمل محدد، ثم تعيد تحديثاً للحالة.


In [5]:
def classify_intent_node(state: SimpleAgentState) -> Dict[str, Any]:
    query = state["messages"][-1].content.lower()
    
    # فحص الكلمات المفتاحية لتحديد نوع السؤال
    keywords = ["langgraph", "rag", "agent", "python", "ai", "ذكاء", "شرح"]
    is_tech = any(w in query for w in keywords)
    detected_intent = "technical" if is_tech else "greeting"
    
    print(f"[عقدة التصنيف]: تم التعرف على نوع السؤال -> {detected_intent}")
    return {
        "intent": detected_intent,
        "steps_count": state.get("steps_count", 0) + 1
    }


In [6]:
def technical_expert_node(state: SimpleAgentState) -> Dict[str, Any]:
    print("[عقدة الخبير التقني]: جاري إعداد الشرح التقني المتخصص...")
    user_query = state["messages"][-1].content
    
    prompt = [
        SystemMessage(content="أنت مهندس ذكاء اصطناعي خبير. اشرح المفهوم بوضوح وإيجاز علمي دقيق باللغة العربية."),
        HumanMessage(content=user_query)
    ]
    
    if llm:
        reply = llm.invoke(prompt)
    else:
        reply = AIMessage(content="إطار عمل LangGraph يوفر بناء تطبيقات وكلاء الحالة Stateful Multi-Actor Graphs.")
        
    return {
        "messages": [reply],
        "steps_count": state["steps_count"] + 1
    }


In [7]:
def friendly_greeter_node(state: SimpleAgentState) -> Dict[str, Any]:
    print("[عقدة الترحيب]: جاري إرسال رد ترحيبي ودود...")
    reply = AIMessage(content="أهلاً بك يا بطل! كيف أستطيع مساعدتك اليوم في رحلتك مع الذكاء الاصطناعي؟")
    return {
        "messages": [reply],
        "steps_count": state["steps_count"] + 1
    }


### 4. دالة التوجيه الشرطي (Conditional Routing)
هذه الدالة تقرأ الحالة وتقرر: هل نذهب لعقدة الخبير التقني أم عقدة الترحيب؟


In [8]:
def decide_next_node(state: SimpleAgentState) -> Literal["technical_expert", "friendly_greeter"]:
    if state["intent"] == "technical":
        return "technical_expert"
    return "friendly_greeter"

print("تم تجهيز دالة التوجيه الشرطي بنجاح!")


تم تجهيز دالة التوجيه الشرطي بنجاح!


### 5. تجميع الجراف وتوصيل العقد
الآن سنربط العقد مع بعضها ونضيف ذاكرة حفظ الحالات (MemorySaver).


In [9]:
builder = StateGraph(SimpleAgentState)

# 1. إضافة العقد
builder.add_node("classify_intent", classify_intent_node)
builder.add_node("technical_expert", technical_expert_node)
builder.add_node("friendly_greeter", friendly_greeter_node)

# 2. ربط نقطة البداية بعقدة التصنيف
builder.add_edge(START, "classify_intent")

# 3. إضافة التفرع الشرطي
builder.add_conditional_edges(
    "classify_intent",
    decide_next_node,
    {
        "technical_expert": "technical_expert",
        "friendly_greeter": "friendly_greeter"
    }
)

# 4. توصيل نهايات المسارين بنقطة النهاية END
builder.add_edge("technical_expert", END)
builder.add_edge("friendly_greeter", END)

# 5. التجميع مع الذاكرة
checkpointer = MemorySaver()
agent_app = builder.compile(checkpointer=checkpointer)

print("تم تجميع الجراف بنجاح وهو جاهز للتشغيل!")


تم تجميع الجراف بنجاح وهو جاهز للتشغيل!


### 6. تجربة تشغيل سؤال تقني
تعال نجرب سؤالاً تقنياً ونشاهد كيف يتحرك الجراف عبر العقد.


In [10]:
config_tech = {"configurable": {"thread_id": "session-arabic-tech-1"}}
tech_input = {
    "messages": [HumanMessage(content="ما هي أهمية استخدام LangGraph في بناء أنظمة الوكلاء؟")]
}

print("بدء تنفيذ الجراف على السؤال التقني:")
print("-" * 50)
for output in agent_app.stream(tech_input, config=config_tech):
    for node, data in output.items():
        print(f"العقدة المنفذة حالياً: [{node}]")

print("-" * 50)
final_state = agent_app.get_state(config_tech)
print("الرد النهائي للخبير التقني:")
print(final_state.values["messages"][-1].content)


بدء تنفيذ الجراف على السؤال التقني:
--------------------------------------------------
[عقدة التصنيف]: تم التعرف على نوع السؤال -> technical
العقدة المنفذة حالياً: [classify_intent]
[عقدة الخبير التقني]: جاري إعداد الشرح التقني المتخصص...


العقدة المنفذة حالياً: [technical_expert]
--------------------------------------------------
الرد النهائي للخبير التقني:
**أهمية استخدام LangGraph في بناء أنظمة الوكلاء**

| الجوانب | لماذا يُعد LangGraph حيوياً؟ |
|--------|---------------------------|
| **تصميم سير العمل كـ “مخطط رسومي”** | يتيح تمثيل كل خطوة (الاستعلام، المعالجة، التخزين، التفاعل) كعقد متصلة، ما يسهل تصور وتعديل تدفق العمل. |
| **إدارة الحالة (State) المتسقة** | كل عقدة تُحافظ على حالة محدّثة للبيانات، مما يقلل الأخطاء ويُسهل تتبع التغييرات عبر الزمن. |
| **التكامل مع LLMs** | يوفر واجهة موحدة لاستدعاء نماذج اللغة، مع إمكانية تخصيص المعلمات (temperature, top_p) لكل عقدة. |
| **التعامل مع السياق والذاكرة** | يدعم تخزين السياق طويل الأمد (memory) وتحديثه ديناميكياً، ما يتيح للوكيل “تذكر” المحادثات السابقة. |
| **التوسع والتكرار** | يمكن إضافة أو حذف عقدة دون إعادة بناء النظام بأكمله؛ يُسهل التوسع الأفقي (أكثر وحدات) أو الرأسي (تحسينات). |
| **الاختبار والتحليل** | يمكن تشغيل كل عقدة بشكل مستقل أو معاً، ما يُسهل اختبار

### 7. تجربة تشغيل تحية عادية
الآن تعال نجرب رسالة ترحيبية بسيطة لنرى كيف يختار الجراف المسار الآخر تماماً.


In [11]:
config_greet = {"configurable": {"thread_id": "session-arabic-greet-1"}}
greet_input = {
    "messages": [HumanMessage(content="مرحباً، صباح الخير!")]
}

print("بدء تنفيذ الجراف على رسالة الترحيب:")
print("-" * 50)
for output in agent_app.stream(greet_input, config=config_greet):
    for node, data in output.items():
        print(f"العقدة المنفذة حالياً: [{node}]")

print("-" * 50)
greet_state = agent_app.get_state(config_greet)
print("الرد الترحيبي النهائي:")
print(greet_state.values["messages"][-1].content)


بدء تنفيذ الجراف على رسالة الترحيب:
--------------------------------------------------
[عقدة التصنيف]: تم التعرف على نوع السؤال -> greeting
العقدة المنفذة حالياً: [classify_intent]
[عقدة الترحيب]: جاري إرسال رد ترحيبي ودود...
العقدة المنفذة حالياً: [friendly_greeter]
--------------------------------------------------
الرد الترحيبي النهائي:
أهلاً بك يا بطل! كيف أستطيع مساعدتك اليوم في رحلتك مع الذكاء الاصطناعي؟


### 8. استعراض الذاكرة ونقاط التفتيش (Time-Travel)
بفضل MemorySaver، يسجل الجراف كل خطوة فائقة (Superstep) تمت في الجلسة.
تعال نستعرض تاريخ الجلسة التقنية:


In [12]:
history_snapshots = list(agent_app.get_state_history(config_tech))
print(f"عدد نقاط التفتيش المسجلة في الجلسة: {len(history_snapshots)}")

for i, snap in enumerate(reversed(history_snapshots)):
    target = snap.next if snap.next else ("نهاية الجلسة END",)
    print(f"الخطوة {i}: المسار القادم -> {target}")


عدد نقاط التفتيش المسجلة في الجلسة: 4
الخطوة 0: المسار القادم -> ('__start__',)
الخطوة 1: المسار القادم -> ('classify_intent',)
الخطوة 2: المسار القادم -> ('technical_expert',)
الخطوة 3: المسار القادم -> ('نهاية الجلسة END',)


### 9. جرب بنفسك! (Hands-on Experiments)
والآن جاء دورك لتجربة وتطوير الجراف بنفسك:
1. **تحدي 1**: أضف عقدة جديدة باسم `code_explainer_node` متخصصة في شرح أكواد بايثون فقط.
2. **تحدي 2**: عدل دالة `decide_next_node` بحيث إذا كان السؤال يحتوي كلمة "كود" أو "code"، يوجه الجراف لعقدتك الجديدة.
3. **تحدي 3**: جرب إضافة متغير جديد في الحالة `SimpleAgentState` مثل `user_language: str` واجعله يتغير ديناميكياً بناءً على لغة المستخدم!
